Question 6.2:

In [2]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

In [3]:
TARGET_COL = "price"
DROP_COLS = ["id", "date", "zipcode", "Unnamed: 0"]

def load_prepare(path: str):
    df = pd.read_csv(path)

    y = df[TARGET_COL].astype(float).to_numpy() / 1000.0
    X = df.drop(columns=[TARGET_COL] + [c for c in DROP_COLS if c in df.columns], errors="ignore")
    X = X.select_dtypes(include = [np.number])
    return X , y

def add_intercept(X: np.ndarray):
    return np.column_stack([np.ones(X.shape[0]), X])

def ridge_gradient_descent(X: np.ndarray, y: np.ndarray, alpha: float, lam: float, num_iters: int):
    #Objective (common form):
    #J(theta) = (1/(2m)) * ||X_aug theta - y||^2 + (lam/(2m)) * sum_{j>=1} theta_j^2
    #Gradient:
    #grad = (1/m) * X_aug^T (X_aug theta - y)
    #grad[j] += (lam/m) * theta[j] for j>=1

    m, d = X.shape
    X_aug = add_intercept(X)
    theta = np.zeros(d + 1)

    for _ in range(num_iters):
        preds = X_aug @ theta
        grad = (1 / m) * (X_aug.T @ (preds - y))

        # ridge penalty
        grad[1:] += (lam / m) * theta[1:]
        theta = theta - alpha * grad
    return theta

def predict(X: np.ndarray, theta: np.ndarray):
    return add_intercept(X) @ theta

def evaluate(y_true, y_pred):
    return mean_squared_error(y_true, y_pred), r2_score(y_true, y_pred)


In [4]:
train_path = "train.csv"
test_path  = "test.csv"

X_train_df, y_train = load_prepare(train_path)
X_test_df,  y_test  = load_prepare(test_path)

X_train_df, X_test_df = X_train_df.align(X_test_df, join = "inner", axis = 1)
feature_names = X_train_df.columns.tolist()
imputer = SimpleImputer(strategy = "median")
scaler = StandardScaler()

X_train = scaler.fit_transform(imputer.fit_transform(X_train_df))
X_test  = scaler.transform(imputer.transform(X_test_df))

alphas = [0.01, 0.1, 0.5]
iters_list = [10, 50, 100]

lambdas = [0, 1, 10, 100]
rows = []
thetas = {}

for lam in lambdas:
    for a in alphas:
        for iters in iters_list:
            theta = ridge_gradient_descent(X_train, y_train, a, lam, iters)
            thetas[(lam, a, iters)] = theta
            pred_tr = predict(X_train, theta)
            pred_te = predict(X_test, theta)
            tr_mse, tr_r2 = evaluate(y_train, pred_tr)
            te_mse, te_r2 = evaluate(y_test , pred_te)
            rows.append([lam, a, iters, tr_mse, tr_r2, te_mse, te_r2])

results = pd.DataFrame(
    rows,
    columns=["lambda", "alpha", "iters", "Train MSE", "Train R2", "Test MSE", "Test R2"]
)
print(results)
print("\nParameter order:", ["intercept"] + feature_names)
for lam in lambdas:
    for a in alphas:
        for iters in iters_list:
            print(f"\nlambda={lam}, alpha={a}, iters={iters}")
            print(thetas[(lam, a, iters)])

    lambda  alpha  iters     Train MSE      Train R2      Test MSE  \
0        0   0.01     10  2.947987e+05 -1.560413e+00  3.505251e+05   
1        0   0.01     50  1.382959e+05 -2.011404e-01  1.703767e+05   
2        0   0.01    100  7.011899e+04  3.909961e-01  9.748624e+04   
3        0   0.10     10  6.649932e+04  4.224340e-01  9.355929e+04   
4        0   0.10     50  3.157898e+04  7.257273e-01  5.801232e+04   
5        0   0.10    100  3.149769e+04  7.264333e-01  5.772519e+04   
6        0   0.50     10  6.118299e+08 -5.312921e+03  6.850231e+08   
7        0   0.50     50  1.649496e+25 -1.432635e+20  1.842083e+25   
8        0   0.50    100  5.698752e+45 -4.949532e+40  6.364111e+45   
9        1   0.01     10  2.948001e+05 -1.560425e+00  3.505272e+05   
10       1   0.01     50  1.383003e+05 -2.011785e-01  1.703860e+05   
11       1   0.01    100  7.012293e+04  3.909618e-01  9.749681e+04   
12       1   0.10     10  6.650278e+04  4.224039e-01  9.356943e+04   
13       1   0.10   

Question 6.3

In [5]:
np.random.seed(0)
N = 1000
X = np.random.uniform(-2, 2, size=N)
e = np.random.normal(0, np.sqrt(2), size=N)   # N(0,2) treated as variance 2
y = 1 + 2*X + e

X_aug = np.column_stack([np.ones(N), X])
XtX = X_aug.T @ X_aug
Xty = X_aug.T @ y

def fit_ridge_closed_form(lam):
    R = np.diag([0, lam])
    w = np.linalg.solve(XtX + R, Xty)
    return w
w_ols = np.linalg.pinv(X_aug) @ y
pred_ols = X_aug @ w_ols

rows = []
rows.append(["OLS", 0, w_ols[1], mean_squared_error(y, pred_ols), r2_score(y, pred_ols)])

for lam in [1, 10, 100, 1000, 10000]:
    w = fit_ridge_closed_form(lam)
    pred = X_aug @ w
    rows.append(["Ridge", lam, w[1], mean_squared_error(y, pred), r2_score(y, pred)])

df = pd.DataFrame(rows, columns=["Model", "lambda", "slope", "MSE", "R2"])
print(df.to_string(index=False))

Model  lambda    slope      MSE       R2
  OLS       0 1.965692 1.865374 0.736759
Ridge       1 1.964238 1.865377 0.736759
Ridge      10 1.951251 1.865656 0.736720
Ridge     100 1.830236 1.890166 0.733261
Ridge    1000 1.129641 2.809812 0.603481
Ridge   10000 0.233982 5.917267 0.164958


As the regularization parameter λ increases, the ridge penalty gets stronger and it shrinks the slope towards 0. We can see that here because the slope is about 1.97 for OLS and λ = 1, 10, then it drops to 1.83 at λ = 100, 1.13 at λ = 1000 and all the way to 0.234 at λ = 10000. For small λ values (1 and 10), ridge acts almost the same as OLS, so MSE and R^2 barely change. As λ becomes large, the model starts to underfit because the coefficient has to be small, so the fit becomes worse-- MSE increases and R^2 decreases. Very large λ makes the model close to predicting a constant value which explains the large error and low R^2.